In [15]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langgraph.checkpoint.memory import MemorySaver

In [4]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage] , add_messages]

In [5]:
model = ChatOpenAI()

In [7]:
def chat_bot(state : ChatState):
    # get the user input
    messages = state['messages']

    #  invoke llm 
    response = model.invoke(messages)

    #  return response
    return {'messages' : [response]}

In [8]:
graph = StateGraph(ChatState)

graph.add_node('chat_bot',chat_bot)

graph.add_edge(START,'chat_bot')
graph.add_edge('chat_bot',END)

chatbot = graph.compile()

In [13]:
initial_state = {
    'messages' : HumanMessage(content = "Who is India's best cricekter ?")
}

chatbot.invoke(initial_state)['messages'][-1].content

'There are many outstanding cricketers in India, but some of the most notable and widely regarded as the best include players like Virat Kohli, Sachin Tendulkar, Sunil Gavaskar, Kapil Dev, and MS Dhoni. Each of these players has made significant contributions to Indian cricket and achieved great success in their careers. It is ultimately subjective to decide who is the best cricketer as it can vary based on personal opinions and criteria.'

# The biggest problem with above chatbot is that the state is not stored. Everytime when we call .invoke function the state is starting from fresh. Hence the state is not persisted.

# Chatbot -> With While Loop

In [14]:
while True :

    user_message = input("Please type here : ")

    print("User : ",user_message)

    if user_message.strip().lower() in ['quit','bye','exit'] :
        break

    response = chatbot.invoke({'messages' : HumanMessage(content=user_message)})

    final_response = response['messages'][-1].content

    print("AI : " , final_response)


User :  Hi My name is Avanindra
AI :  Hello Avanindra! How can I assist you today?
User :  what is 10 _ 25
AI :  10 - 25 = -15
User :  add + 98 in the previous response
AI :  That would make the new total 282.
User :  quit


# with a basic Persistance of state in RAM

In [16]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node('chat_bot',chat_bot)

graph.add_edge(START,'chat_bot')
graph.add_edge('chat_bot',END)

chatbot_optimised = graph.compile(checkpointer=checkpointer)

In [17]:
thread_id = '1'

while True :

    user_message = input("Please type here : ")

    print("User : ",user_message)

    if user_message.strip().lower() in ['quit','bye','exit'] :
        break

    config = {'configurable': {'thread_id': thread_id}}

    response = chatbot_optimised.invoke({'messages' : HumanMessage(content=user_message)},config=config)

    final_response = response['messages'][-1].content

    print("AI : " , final_response)

User :  Hi I am Avanindra
AI :  Hello Avanindra! How can I assist you today?
User :  What is my name ?
AI :  Your name is Avanindra, as you mentioned earlier. How can I help you today, Avanindra?
User :  What is 2 * 78
AI :  2 multiplied by 78 is equal to 156.
User :  Subtract 50 from this
AI :  By subtracting 50 from 156, you would have 106.
User :  exit


In [18]:
chatbot_optimised.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='Hi I am Avanindra', additional_kwargs={}, response_metadata={}, id='b771acd5-b8f1-496f-bd89-19eda0b27a0a'), AIMessage(content='Hello Avanindra! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 14, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EDAIGJTZ0LlXND5h1gRAdTuBtdkQ9', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a005f8-1793-7ac2-b53f-9c6bcba57675-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 13, 'total_tokens': 27, 'input_token_detai